# SAFOD DAS incremental-value development protocol

This notebook is a compact protocol for the next computation. It is not a result notebook and it does not run the continuous detectors.

The frozen 50-minute development interval is nonblind and may be used to make both pipelines operational. Network-only must be frozen first. DAS-only must generate candidates without seeing network trigger times. The 12 held-out hours remain sealed until both pipelines, the event definition, false-discovery controls, and adjudication rules are frozen.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT = next(
    (
        path.resolve()
        for path in search_roots
        if (path / "config" / "incremental_value.json").exists()
    ),
    None,
)
if PROJECT is None:
    raise FileNotFoundError(
        "Could not locate the standalone safod-das-repeaters repository"
    )

with (PROJECT / "config" / "incremental_value.json").open(
    "r", encoding="utf-8"
) as handle:
    config = json.load(handle)
with (
    PROJECT / "outputs" / "checkpoint" / "advisor_checkpoint.json"
).open("r", encoding="utf-8") as handle:
    checkpoint = json.load(handle)

print("Project:", PROJECT)
print("Decision:", checkpoint["project_decision"])
print("This notebook opens compact tables only.")

## 1. Frozen development population

The 2025-01-20 04:55--05:45 UTC interval was separated before the held-out draw. Official catalog events are context and adjudication aids; neither detector may be reduced to replaying their origin times. Two location nominees fall inside these 50 minutes, and both have already been rejected by the frozen target-family verifier.

In [ ]:
INCREMENTAL = PROJECT / "outputs" / "incremental_value"
development = config["development_interval"]
catalog = pd.read_csv(
    INCREMENTAL / "ncedc_archive_catalog.csv", dtype={"event_id": str}
)
catalog["origin_time"] = pd.to_datetime(catalog["origin_time"], utc=True)
start = pd.Timestamp(development["start_utc"])
end = pd.Timestamp(development["end_utc"])
development_catalog = catalog.loc[
    (catalog["origin_time"] >= start) & (catalog["origin_time"] < end)
].copy()
network_decisions = pd.read_csv(
    INCREMENTAL
    / "prospective_network"
    / "frozen_network_decisions.csv",
    dtype={"event_id": str},
)
development_view = development_catalog.merge(
    network_decisions[
        [
            "event_id",
            "median_target_correlation",
            "median_target_differential_lag_rms_s",
            "frozen_decision",
        ]
    ],
    on="event_id",
    how="left",
)

display(
    pd.DataFrame(
        [
            {
                "interval": "development",
                "start UTC": development["start_utc"],
                "end UTC": development["end_utc"],
                "duration minutes": (end - start).total_seconds() / 60.0,
                "selection role": development["selection_role"],
                "official events in interval": len(development_catalog),
            }
        ]
    )
)
display(development_view)

## 2. Information firewall and deliverables

Candidate generation and family scoring are separate. A detector may nominate an event without assigning it to a repeater family. An unavailable or ambiguous waveform is an abstention, not a negative label.

In [ ]:
pipeline_contract = pd.DataFrame(
    [
        {
            "pipeline": "network only",
            "candidate input": "continuous BP array, NC.PSM, and BK.PKD",
            "forbidden input": "DAS triggers, DAS scores, DAS event times",
            "required output": "candidate UTC, score, station support, null/FDR provenance",
            "freeze order": 1,
        },
        {
            "pipeline": "DAS only",
            "candidate input": "continuous primary-configuration DAS",
            "forbidden input": "network triggers, network scores, routine-catalog replay",
            "required output": "candidate UTC, channel/block coherence, null/FDR provenance",
            "freeze order": 2,
        },
        {
            "pipeline": "joint",
            "candidate input": "union of two frozen score tables",
            "forbidden input": "held-out truth-driven retuning",
            "required output": "predeclared fusion score and abstention reason",
            "freeze order": 3,
        },
    ]
)
display(pipeline_contract)

## 3. Event-level evaluation contract

A file-level trigger is not an event. Detections must be deduplicated across adjacent files, channels, templates, and stations before false-discovery rate is computed. The blind union must include pipeline-only candidates, and adjudicators should not see which pipeline nominated a candidate.

In [ ]:
metric_contract = pd.DataFrame(
    [
        {
            "claim": "detection extension",
            "primary metric": "event recall difference at matched event-level FDR",
            "uncertainty": "interval/block bootstrap",
            "hard failure": "gain disappears after deduplication or matched-FDR control",
        },
        {
            "claim": "classification extension",
            "primary metric": "event-held-out macro F1 and cross-family merge rate",
            "uncertainty": "event/block bootstrap when identifiable",
            "hard failure": "apparent gain requires treating one disputed catalog as truth",
        },
        {
            "claim": "near-source resolution",
            "primary metric": "within-versus-between partition effect",
            "uncertainty": "channel, band, polarity, and time-block sensitivity",
            "hard failure": "effect depends on unsurveyed depth/source-distance mapping",
        },
    ]
)
display(metric_contract)

## 4. Held-out intervals remain sealed

These times are visible because the detectors must know their eventual data scope, but no waveform-derived statistic from them may influence development. Their selection used DAS manifest coverage and a frozen seed only.

In [ ]:
heldout = pd.read_csv(INCREMENTAL / "heldout_intervals.csv")
display(
    heldout[
        [
            "interval_id",
            "start_utc",
            "end_utc",
            "coverage_segment_id",
            "selection_seed",
            "selection_inputs",
            "analysis_status",
        ]
    ]
)
assert heldout["analysis_status"].eq("SEALED_NOT_RUN").all()
print("Seal check: all {} intervals remain SEALED_NOT_RUN".format(len(heldout)))

## 5. Implementation checkpoint

Before any held-out run, the repository must contain:

1. a network continuous-detection script and frozen development output;
2. a DAS-only continuous-detection script that cannot import network candidates;
3. injection/null recovery curves and an event-deduplication test for each;
4. a blinded union/adjudication table schema;
5. configuration and code hashes for every frozen stage; and
6. an advisor-readable development report showing failures as well as passes.

Stress drop and creep-rate code are intentionally outside this checkpoint.

In [ ]:
display(pd.DataFrame(checkpoint["project_shape"]))
print("Registered next analysis:")
print(checkpoint["highest_value_next_analysis"])
print()
print("No raw waveforms or held-out statistics were opened.")